# Fake News Classification with BiLSTM — Dataset 2

This notebook uses the repository's Fake News Detection dataset with two columns: `text` and binary `label`. The supplied test file is held out for final evaluation.

**Label convention:** 0 = Fake, 1 = Real  
**Input text:** `text`  
**Model:** Bidirectional LSTM

In [ ]:
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", len(tf.config.list_physical_devices("GPU")))

## 1. Load the dataset

In [ ]:
DATA_DIR_CANDIDATES = [
    Path("../data/dataset_2_fake_news_detection"),
    Path("data/dataset_2_fake_news_detection"),
]
DATA_DIR = next((p.resolve() for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset 2 folder not found. Start Jupyter from the repository root "
        "or from the notebooks directory."
    )

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Train or test CSV file not found.")

train_df = pd.read_csv(train_path, sep=";")
test_df = pd.read_csv(test_path, sep=";")

required_columns = {"text", "label"}
for name, frame in [("train", train_df), ("test", test_df)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{name}.csv is missing required columns: {sorted(missing)}")

train_df = train_df.dropna(subset=["text", "label"]).copy()
test_df = test_df.dropna(subset=["text", "label"]).copy()

train_df["label"] = pd.to_numeric(train_df["label"], errors="raise").astype(int)
test_df["label"] = pd.to_numeric(test_df["label"], errors="raise").astype(int)

if not set(train_df["label"].unique()).issubset({0, 1}):
    raise ValueError("Training labels must be binary: 0 (Fake), 1 (Real).")
if not set(test_df["label"].unique()).issubset({0, 1}):
    raise ValueError("Test labels must be binary: 0 (Fake), 1 (Real).")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())
display(train_df.head())

## 2. Explore label distribution

In [ ]:
sns.countplot(data=train_df, x="label")
plt.title("Training data: Fake vs Real")
plt.xticks([0, 1], ["Fake", "Real"])
plt.xlabel("News label")
plt.ylabel("Count")
plt.show()

print("Label distribution:")
display(train_df["label"].value_counts().sort_index())

## 3. Preprocess text and split the data

The tokenizer is fitted only on the training subset. The supplied test file is not used during training or validation.

In [ ]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    tokens = re.sub(r"[^a-z]", " ", text).split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]
    return " ".join(tokens)

X_train_full_text = pd.Series(
    [clean_text(text) for text in train_df["text"]],
    index=train_df.index
)
X_test_text = pd.Series(
    [clean_text(text) for text in test_df["text"]],
    index=test_df.index
)

y_train_full = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_full_text,
    y_train_full,
    test_size=0.1765,
    random_state=SEED,
    stratify=y_train_full
)

print("Train:", len(X_train_text), "Validation:", len(X_val_text), "Test:", len(X_test_text))

## 4. Tokenization and padding

In [ ]:
MAX_VOCAB_SIZE = 5000
MAX_SEQUENCE_LENGTH = 400
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

def encode_texts(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(
        sequences,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="pre",
        truncating="pre",
    )

X_train = encode_texts(X_train_text)
X_val = encode_texts(X_val_text)
X_test = encode_texts(X_test_text)

print("Encoded train shape:", X_train.shape)
print("Vocabulary size used:", min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE))

## 5. Build and train the BiLSTM model

In [ ]:
inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))
x = Embedding(
    input_dim=MAX_VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
)(inputs)
x = Bidirectional(LSTM(150))(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1,
)

## 6. Training curves

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["accuracy"], label="Train")
axes[0].plot(history_df["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["loss"], label="Train")
axes[1].plot(history_df["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Final test-set evaluation

The final metrics below are calculated once on the held-out test file.

In [ ]:
test_probabilities = model.predict(X_test, verbose=0).ravel()
y_pred = (test_probabilities >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fake", "Real"],
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
)
plt.title("Confusion matrix — Dataset 2")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Notes

- This version uses the supplied train/test split and creates validation data from the training file only. Historical metrics may therefore change.
- The original label convention is preserved: 0 = Fake, 1 = Real.
- Training can take a while on CPU.
- Run the notebook from the repository root or from the `notebooks/` directory.